# Multi-Task LSTM Model Training
This notebook builds and trains a multi-task LSTM model to predict energy demand, detect outages, and classify system alerts.

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
import time
from pathlib import Path
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
import os

# Settings
os.makedirs('../models/trained', exist_ok=True)
print(f'TensorFlow Version: {tf.__version__}')

TensorFlow Version: 2.21.0


## Section 1 — Load Data & Oversample Minority Classes

In [3]:
prep_dir = Path('../models/preprocessing')

X_train = np.load(prep_dir / 'X_train.npy')
y_demand_train = np.load(prep_dir / 'y_demand_train.npy')
y_anomaly_train = np.load(prep_dir / 'y_anomaly_train.npy')
y_alert_train = np.load(prep_dir / 'y_alert_train.npy')

X_val = np.load(prep_dir / 'X_val.npy')
y_demand_val = np.load(prep_dir / 'y_demand_val.npy')
y_anomaly_val = np.load(prep_dir / 'y_anomaly_val.npy')
y_alert_val = np.load(prep_dir / 'y_alert_val.npy')

# Fix 1 — Oversampling
outage_idx = np.where(y_anomaly_train == 1)[0]
critical_idx = np.where(y_alert_train == 2)[0]
minority_idx = np.unique(np.concatenate([outage_idx, critical_idx]))

print(f'Original anomaly samples: {len(outage_idx)}')
print(f'Original critical samples: {len(critical_idx)}')

X_extra = X_train[minority_idx]
yd_extra = y_demand_train[minority_idx]
ya_extra = y_anomaly_train[minority_idx]
yl_extra = y_alert_train[minority_idx]

X_train = np.concatenate([X_train] + [X_extra]*20)
y_demand_train = np.concatenate([y_demand_train] + [yd_extra]*20)
y_anomaly_train = np.concatenate([y_anomaly_train] + [ya_extra]*20)
y_alert_train = np.concatenate([y_alert_train] + [yl_extra]*20)

# Shuffle
idx = np.arange(len(X_train))
np.random.seed(42)
np.random.shuffle(idx)
X_train, y_demand_train, y_anomaly_train, y_alert_train = \
    X_train[idx], y_demand_train[idx], y_anomaly_train[idx], y_alert_train[idx]

print(f'New training shape: {X_train.shape}')
print(f'New anomaly samples: {np.sum(y_anomaly_train)}')
print(f'New critical samples: {np.sum(y_alert_train == 2)}')

Original anomaly samples: 337
Original critical samples: 108
New training shape: (31212, 48, 21)
New anomaly samples: 7077
New critical samples: 2268


## Section 2 — Model Architecture

In [4]:
input_shape = (48, 21)
inputs = layers.Input(shape=input_shape, name='main_input')
x = layers.LSTM(128, return_sequences=True)(inputs)
x = layers.Dropout(0.2)(x)
x = layers.LSTM(64, return_sequences=False)(x)
x = layers.Dropout(0.2)(x)
shared = layers.Dense(32, activation='relu', name='shared_dense')(x)

demand_output = layers.Dense(16, activation='relu')(shared)
demand_output = layers.Dense(8, activation='linear', name='demand_output')(demand_output)
anomaly_output = layers.Dense(8, activation='relu')(shared)
anomaly_output = layers.Dense(1, activation='sigmoid', name='anomaly_output')(anomaly_output)
alert_output = layers.Dense(16, activation='relu')(shared)
alert_output = layers.Dense(3, activation='softmax', name='alert_output')(alert_output)

model = models.Model(inputs=inputs, outputs=[demand_output, anomaly_output, alert_output])
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ main_input (InputLayer)       │ (None, 48, 21)            │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ lstm (LSTM)                   │ (None, 48, 128)           │          76,800 │ main_input[0][0]           │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dropout (Dropout)             │ (None, 48, 128)           │               0 │ lstm[0][0]                 │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ lstm_1 (LSTM)                 │ (None, 64)                │          49,408 │ dropout[0][0]              │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dropout_1 (Dropout)           │ (None, 64)                │               0 │ lstm_1[0][0]               │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ shared_dense (Dense)          │ (None, 32)                │           2,080 │ dropout_1[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dense (Dense)                 │ (None, 16)                │             528 │ shared_dense[0][0]         │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dense_1 (Dense)               │ (None, 8)                 │             264 │ shared_dense[0][0]         │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dense_2 (Dense)               │ (None, 16)                │             528 │ shared_dense[0][0]         │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ demand_output (Dense)         │ (None, 8)                 │             136 │ dense[0][0]                │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ anomaly_output (Dense)        │ (None, 1)                 │               9 │ dense_1[0][0]              │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ alert_output (Dense)          │ (None, 3)                 │              51 │ dense_2[0][0]              │
└───────────────────────────────┴───────────────────────────┴─────────────────┴────────────────────────────┘

 Total params: 129,804 (507.05 KB)

 Trainable params: 129,804 (507.05 KB)

 Non-trainable params: 0 (0.00 B)

## Section 3 — Compile (Updated Loss Weights)

In [5]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0005), # Fix 4
    loss={
        'demand_output': 'mse',
        'anomaly_output': 'binary_crossentropy',
        'alert_output': 'sparse_categorical_crossentropy'
    },
    loss_weights={
        'demand_output': 0.5,
        'anomaly_output': 4.0,
        'alert_output': 4.0
    },
    metrics={
        'demand_output': 'mae',
        'anomaly_output': 'accuracy',
        'alert_output': 'accuracy'
    }
)

## Section 4 — Callbacks

In [6]:
model_callbacks = [
    callbacks.EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True),
    callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6),
    callbacks.ModelCheckpoint('../models/trained/best_model.keras', monitor='val_loss', save_best_only=True),
    callbacks.CSVLogger('../models/trained/training_log.csv')
]

## Section 5 — Train (Per-Sample Weights)

In [7]:
# Fix 3 — Per-sample weights
anomaly_weights = np.where(y_anomaly_train == 1, 30.0, 1.0)
alert_weights = np.where(y_alert_train == 2, 30.0, np.where(y_alert_train == 1, 5.0, 1.0))

sample_weights = {
    'demand_output': np.ones(len(y_demand_train)),
    'anomaly_output': anomaly_weights,
    'alert_output': alert_weights
}

start_time = time.time()
history = model.fit(
    X_train,
    [y_demand_train, y_anomaly_train, y_alert_train],
    validation_data=(X_val, [y_demand_val, y_anomaly_val, y_alert_val]),
    epochs=150,
    batch_size=64,
    sample_weight=sample_weights,
    callbacks=model_callbacks,
    verbose=1
)

end_time = time.time()
print(f'Training finished at {time.ctime()}')
best_epoch = np.argmin(history.history['val_loss'])
print(f'Best epoch: {best_epoch + 1} with val_loss: {history.history["val_loss"][best_epoch]:.4f}')

ValueError: You should provide one `sample_weight` array per output in `y`. The two structures did not match:
- y: [array([[159.57331059, 157.55282727, 162.89689405, ..., 162.00481715,
        160.85547757, 157.54699268],
       [307.1776519 , 304.85692544, 303.45531279, ..., 316.73433222,
        316.86666448, 310.06721376],
       [347.69082877, 285.19149686, 285.58255768, ..., 178.89241355,
        178.12425236, 177.84008142],
       ...,
       [187.27256607, 176.75399323, 174.32990088, ..., 176.73547251,
        177.02446229, 179.40797487],
       [160.06091914, 157.49718906, 158.01880011, ..., 156.0744315 ,
        157.81704783, 154.81777409],
       [264.39746064, 253.040147  , 251.41250512, ..., 202.42757539,
        162.14419652, 158.66979646]]), array([0, 0, 0, ..., 0, 0, 0]), array([0, 0, 0, ..., 0, 0, 0], dtype=int64)]
- sample_weight: {'demand_output': array([1., 1., 1., ..., 1., 1., 1.]), 'anomaly_output': array([1., 1., 1., ..., 1., 1., 1.]), 'alert_output': array([1., 1

## Section 6 — Training Curves

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 12))
best_epoch = np.argmin(history.history['val_loss'])

# Total Loss
axes[0,0].plot(history.history['loss'], label='Train')
axes[0,0].plot(history.history['val_loss'], label='Val')
axes[0,0].axvline(best_epoch, color='black', linestyle='--')
axes[0,0].set_title('Total Loss')
axes[0,0].legend()

# Demand MAE
axes[0,1].plot(history.history['demand_output_mae'], label='Train')
axes[0,1].plot(history.history['val_demand_output_mae'], label='Val')
axes[0,1].axvline(best_epoch, color='black', linestyle='--')
axes[0,1].set_title('Demand MAE')
axes[0,1].legend()

# Anomaly Accuracy
axes[1,0].plot(history.history['anomaly_output_accuracy'], label='Train')
axes[1,0].plot(history.history['val_anomaly_output_accuracy'], label='Val')
axes[1,0].axvline(best_epoch, color='black', linestyle='--')
axes[1,0].set_title('Anomaly Accuracy')
axes[1,0].legend()

# Alert Accuracy
axes[1,1].plot(history.history['alert_output_accuracy'], label='Train')
axes[1,1].plot(history.history['val_alert_output_accuracy'], label='Val')
axes[1,1].axvline(best_epoch, color='black', linestyle='--')
axes[1,1].set_title('Alert Accuracy')
axes[1,1].legend()

plt.tight_layout()
plt.savefig('figures/training_curves.png')
plt.show()

## Section 7 — Save

In [ ]:
model.save('../models/trained/final_model.keras')

with open('../models/trained/history.json', 'w') as f:
    json.dump({k: [float(v1) for v1 in v] for k, v in history.history.items()}, f)

print('Models and history saved to models/trained/')

print('\nFinal Metrics (Val):')
for k, v in history.history.items():
    if k.startswith('val_'):
        print(f'{k}: {v[best_epoch]:.4f}')